In [1]:
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Load Model

In [19]:
import sys
sys.path.insert(0, '../../../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.2996, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
teacher_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
teacher_model.set_use_module_grafting(False)
teacher_model.set_use_scc_status(False)

In [27]:
outputs = teacher_model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.6582, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [28]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
teacher_model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [29]:
load_path = '../../../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [30]:
teacher_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [31]:
from train_eval_func import eval_loop

In [32]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [33]:
eval_loop(teacher_model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

# 5. Pruning FF

In [34]:
from structure_prune_helper import register_importance_masks, remove_importance_masks

In [35]:
register_importance_masks(model, device, register_heads=False, register_ffn=True)

  Layer 0: Registered masks (3072 neurons)
  Layer 1: Registered masks (3072 neurons)
  Layer 2: Registered masks (3072 neurons)
  Layer 3: Registered masks (3072 neurons)
  Layer 4: Registered masks (3072 neurons)
  Layer 5: Registered masks (3072 neurons)
  Layer 6: Registered masks (3072 neurons)
  Layer 7: Registered masks (3072 neurons)
  Layer 8: Registered masks (3072 neurons)
  Layer 9: Registered masks (3072 neurons)
  Layer 10: Registered masks (3072 neurons)
  Layer 11: Registered masks (3072 neurons)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [36]:
recovery_epochs=3
patience=3

In [37]:
from structure_prune_helper import get_primary_metric, compute_importance_scores, apply_neuron_masking, remaining_layers_stat
from train_eval_func import train
from torch.optim import AdamW

In [38]:
len(model.bert.encoder.layer)

12

In [39]:
def masks_setGrad(model, set_heads=True, set_ffns=True, mode="freeze"):
    if mode not in ["freeze", "unfreeze"]:
        return "Send 'Freeze' or 'Unfreeze' as mode"
    
    for layer in model.bert.encoder.layer:
        if set_ffns:
            layer.int_mask_param.requires_grad = False if mode == 'freeze' else True
            layer.int_mask_param.grad = None
        if set_heads:
            layer.head_mask_param.requires_grad = False if mode == 'freeze' else True
            layer.head_mask_param.grad = None

In [40]:
masks_setGrad(model, set_heads=False, mode="unfreeze")

In [41]:
def iterativeMaskTrain(
    model, dataloader, val_dataloader, device, task_name, recovery_epochs, recovery_lr,  max_batches=None, mask_head=True, mask_ffn=True, 
    num_att_mask=0, num_int_mask=0, num_round=5, patience=3, teacher_model=None, alpha=0.5, temperature=6
):
    # Calculate total number of heads and ffn neurons
    total_heads, total_neurons = 0, 0
    for layer in model.bert.encoder.layer:
        if mask_head:
            total_heads   += len(layer.head_mask_param)
        if mask_ffn:
            total_neurons += len(layer.int_mask_param)

    if mask_head and mask_ffn:
        print(f"Total Number of Heads: {total_heads}, Total Number of FFN Neurons: {total_neurons}")
    elif mask_head:
        print(f"Total Number of Heads: {total_heads}")
    elif mask_ffn:
        print(f"Total Number of FFN Neurons: {total_neurons}")
    else:
        return "No masking operation"
    
    target_metric = get_primary_metric(task_name)
    
    # Original Performance
    print("Evaluating Original Performance")
    orig_metric = eval_loop(model, val_dataloader, task_name, device)[0]
    threshold = orig_metric[target_metric] * 0.01
    
    print("  Metrics:")
    for metric_name, value in orig_metric.items():
        marker = "★" if metric_name == target_metric else " "
        print(f"    {marker} {metric_name}: {value:.4f}")
    print("Threshold:", threshold)

    mask_history = {
        'head_mask_remaining': [],
        'head_mask_state': [],
        'int_mask_remaining': [],
        'int_mask_state': [],
        'effected_metric': []
    }
    for round_idx  in range(num_round):
        print("")
        print("--> Round:", round_idx)

        masks_setGrad(model, set_heads=False, mode="unfreeze")
        
        # Calculate Importance Score
        head_importance, neuron_importance = compute_importance_scores(
            model, dataloader, device, max_batches=max_batches, compute_heads=mask_head,
            compute_ffn= mask_ffn
        )

        masks_setGrad(model, set_heads=False, mode="freeze")

        # Perform Masking
        if mask_head:
            apply_head_masking(model, head_importance, num_att_mask)
            
            remaining_heads = 0
            total_heads     = 0
            for layer in model.bert.encoder.layer:
                remaining_heads += int(sum(layer.head_mask_param))
                total_heads += len(layer.head_mask_param)
            print(f"Remaining Heads: {remaining_heads}, Total Heads: {total_heads}")
            
            head_mask_stats = remaining_layers_stat(model, component='head')
            head_mask_stats_list = []
            print("Head Mask Remaining Stats:")
            for stat in head_mask_stats:
                remain = stat['n_remain'] * 100 / stat['total']
                remain = round(remain, 2)
                print(f"Layer: {stat['layer']}, total: {stat['total']}, Remain: {stat['n_remain']} ({remain}%)")
                stat['mask'] = model.bert.encoder.layer[stat['layer']].head_mask_param.detach().cpu().tolist()
                head_mask_stats_list.append(stat['mask'])
                
            mask_history['head_mask_remaining'].append(remaining_heads)
            mask_history['head_mask_state'].append(head_mask_stats_list)
            
        if mask_ffn:
            apply_neuron_masking(model, neuron_importance, num_int_mask)
            
            remaining_neurons = 0
            total_neurons     = 0
            for layer in model.bert.encoder.layer:
                remaining_neurons += int(sum(layer.int_mask_param))
                total_neurons += len(layer.int_mask_param)
            print(f"Remaining FFN Neurons: {remaining_neurons}, Total FFN Neurons: {total_neurons}")

            int_mask_stats = remaining_layers_stat(model, component='ffn')
            int_mask_stats_list = []
            print("FFN Neuron Mask Remaining Stats:")
            for stat in int_mask_stats:
                remain = stat['n_remain'] * 100 / stat['total']
                remain = round(remain, 2)
                print(f"Layer: {stat['layer']}, total: {stat['total']}, Remain: {stat['n_remain']} ({remain}%)")
                stat['mask'] = model.bert.encoder.layer[stat['layer']].int_mask_param.detach().cpu().tolist()
                int_mask_stats_list.append(stat['mask'])
                
            mask_history['int_mask_remaining'].append(remaining_neurons)
            mask_history['int_mask_state'].append(int_mask_stats_list)
        
        # Effected Performance
        eval_metric = eval_loop(model, val_dataloader, task_name, device)[0]
        print("  Metrics:")
        for metric_name, value in eval_metric.items():
            marker = "★" if metric_name == target_metric else " "
            print(f"    {marker} {metric_name}: {value:.4f}")
        mask_history['effected_metric'].append(eval_metric)

        diff = orig_metric[target_metric] - eval_metric[target_metric]
        if diff > threshold:
            print("  → Recovery training NEEDED")
            
            # Setup optimizer and scheduler
            optimizer = AdamW(model.parameters(), lr=recovery_lr)
            num_training_steps = recovery_epochs * len(train_dataloader)

            print(f"  → Training: {recovery_epochs} epochs, lr={recovery_lr}")

            lr_scheduler   = None
            temp_save_path = None
            train_stats = train(
                model,
                train_dataloader,
                val_dataloader,
                task_name,
                device,
                num_epochs=recovery_epochs,
                lr=recovery_lr,
                patience=patience,
                use_early_stopping=True,
                optimizer=optimizer,
                lr_scheduler=lr_scheduler,
                save_path=temp_save_path,
                display_epoch_iter=True,
                teacher_model=teacher_model, alpha=alpha, temperature=temperature
            )

    return mask_history

In [42]:
num_att_mask = int(12 * 12 * 0.1)
num_att_mask

14

In [43]:
num_int_mask = int(model.bert.encoder.layer[0].intermediate.dense.weight.shape[0] * 12 * 0.1)
num_int_mask

3686

In [44]:
alpha=0.3
temperature=7

In [45]:
mask_history_head = iterativeMaskTrain(
    model, train_dataloader, val_dataloader, device, task_name, max_batches=200, mask_head=False, mask_ffn=True, 
    num_att_mask=num_att_mask, num_int_mask=num_int_mask, num_round=10, recovery_epochs=recovery_epochs,
    recovery_lr=1e-5, patience=patience
)

Total Number of FFN Neurons: 36864
Evaluating Original Performance
  Metrics:
      accuracy: 0.8676
    ★ f1: 0.9094
Threshold: 0.009093959731543625

--> Round: 0


Computing importance (FFN only): 100%|██████████| 115/115 [00:25<00:00,  4.56it/s]



Masking 3686 / 36864 neurons (10.0%)
Remaining FFN Neurons: 33178, Total FFN Neurons: 36864
FFN Neuron Mask Remaining Stats:
Layer: 0, total: 3072, Remain: 3068 (99.87%)
Layer: 1, total: 3072, Remain: 3071 (99.97%)
Layer: 2, total: 3072, Remain: 3072 (100.0%)
Layer: 3, total: 3072, Remain: 3071 (99.97%)
Layer: 4, total: 3072, Remain: 3071 (99.97%)
Layer: 5, total: 3072, Remain: 3068 (99.87%)
Layer: 6, total: 3072, Remain: 3068 (99.87%)
Layer: 7, total: 3072, Remain: 3071 (99.97%)
Layer: 8, total: 3072, Remain: 3054 (99.41%)
Layer: 9, total: 3072, Remain: 2649 (86.23%)
Layer: 10, total: 3072, Remain: 1999 (65.07%)
Layer: 11, total: 3072, Remain: 916 (29.82%)
  Metrics:
      accuracy: 0.8652
    ★ f1: 0.9079

--> Round: 1


Computing importance (FFN only): 100%|██████████| 115/115 [00:24<00:00,  4.76it/s]



Masking 3686 / 36864 neurons (10.0%)
Remaining FFN Neurons: 29492, Total FFN Neurons: 36864
FFN Neuron Mask Remaining Stats:
Layer: 0, total: 3072, Remain: 3062 (99.67%)
Layer: 1, total: 3072, Remain: 3065 (99.77%)
Layer: 2, total: 3072, Remain: 3071 (99.97%)
Layer: 3, total: 3072, Remain: 3065 (99.77%)
Layer: 4, total: 3072, Remain: 3065 (99.77%)
Layer: 5, total: 3072, Remain: 3036 (98.83%)
Layer: 6, total: 3072, Remain: 3041 (98.99%)
Layer: 7, total: 3072, Remain: 3030 (98.63%)
Layer: 8, total: 3072, Remain: 2724 (88.67%)
Layer: 9, total: 3072, Remain: 1363 (44.37%)
Layer: 10, total: 3072, Remain: 696 (22.66%)
Layer: 11, total: 3072, Remain: 274 (8.92%)
  Metrics:
      accuracy: 0.8676
    ★ f1: 0.9094

--> Round: 2


...


<----------------- Epoch 1 ----------------->
Loss: 0.53, Training Metrics:
  accuracy: 0.7754
  f1: 0.8415
Validation Loss: 0.65, Validation Metrics:
  accuracy: 0.7402
  f1: 0.8354
Elapsed Time: 29.2384 sec


Training:  67%|██████▋   | 230/345 [00:56<00:26,  4.40it/s]


<----------------- Epoch 2 ----------------->
Loss: 0.33, Training Metrics:
  accuracy: 0.8566
  f1: 0.8949
Validation Loss: 0.67, Validation Metrics:
  accuracy: 0.7745
  f1: 0.8472
Elapsed Time: 29.1321 sec


Training: 100%|██████████| 345/345 [01:27<00:00,  3.94it/s]


<----------------- Epoch 3 ----------------->
Loss: 0.25, Training Metrics:
  accuracy: 0.8948
  f1: 0.9229
Validation Loss: 0.83, Validation Metrics:
  accuracy: 0.7598
  f1: 0.8435
Elapsed Time: 29.2531 sec

Total Training Time: 87.62 sec


In [46]:
int_mask = []
for i in range(12):
    int_mask.append(model.bert.encoder.layer[0].int_mask_param.tolist())

In [47]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.7598039215686274, 'f1': 0.8434504792332268}

In [48]:
eval_loop(teacher_model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [49]:
def prune_int_dim(model, int_mask):
    """ This method shows how to prune head (remove heads weights) based on
        the head importance scores as described in Michel et al. (http://arxiv.org/abs/1905.10650)
    """
    
    original_num_params = sum(p.numel() for p in model.parameters())
    
    keep_dims_dict = {}
    for layer in range(len(int_mask)):
        keep_dims = []
        for index, value in enumerate(int_mask[layer]):
            if value == 1:
                keep_dims.append(index)
        keep_dims_dict[layer] = keep_dims
    model.prune_int_dims(keep_dims_dict)
    
    pruned_num_params = sum(p.numel() for p in model.parameters())
    
    print("Pruning, original num of params:  %.2e, after pruning %.2e (%.1f percents)" % 
          (original_num_params, pruned_num_params, pruned_num_params / original_num_params * 100))

In [50]:
prune_int_dim(model, int_mask)

Pruning, original num of params:  1.10e+08, after pruning 5.29e+07 (48.3 percents)


/usr/local/lib/python3.10/dist-packages/torch/nn/init.py:582: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


In [51]:
remove_importance_masks(model)

✓ Removed head and FFN masks


In [52]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.5392156862745098, 'f1': 0.5323383084577115}